In [1]:
from __future__ import annotations

"""
Pipeline tạo feature cho bài toán dự báo doanh thu / giá vốn theo ngày.

Mục tiêu của file này:
1. Gom nhiều nguồn dữ liệu giao dịch, vận hành và marketing về cùng grain `Date`.
2. Tạo feature an toàn cho forecasting theo thời gian, hạn chế leakage.
3. Tách rõ:
   - feature tĩnh / profile lịch sử có thể dùng trực tiếp cho tương lai
   - feature autoregressive / recursive cần được cập nhật tuần tự khi suy luận
4. Xuất ra hai bảng:
   - `train_features.csv`: dùng để train / validation
   - `test_features.csv`: dùng cho bước inference sau này

Lưu ý quan trọng:
- Các tín hiệu "same-day" như số order trong chính ngày tương lai là không biết trước.
- Vì vậy pipeline này chỉ dùng dữ liệu raw same-day để học "mẫu lịch sử",
  sau đó loại chúng khỏi bảng feature cuối cùng.
"""

import argparse
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd


# Nhóm target cơ sở mà pipeline dùng để sinh lag / rolling / EWM.
# Giữ gọn:
# - Revenue
# - COGS
# - gross_margin_pct để bắt trạng thái biên lợi nhuận
TARGET_BASE_COLUMNS = ["Revenue", "COGS", "gross_margin_pct"]

# Danh sách độ trễ được dùng cho feature autoregressive.
TARGET_LAGS = [1, 7, 14, 30, 60, 90, 180, 365]

# Cửa sổ rolling dùng để bắt tín hiệu ngắn hạn, trung hạn và quý gần nhất.
TARGET_WINDOWS = [7, 14, 30, 60, 90]

# Span cho EWM, ưu tiên gần hiện tại nhưng vẫn giữ ký ức lịch sử.
TARGET_EWM_SPANS = [7, 14, 30, 60]

# Các khóa lịch để tạo historical profile.
# Ý tưởng: học "mức trung bình lịch sử" theo thứ trong tuần / tháng / ngày trong năm.
PROFILE_KEYSETS: dict[str, list[str]] = {
    "dow": ["dow"],
    "month": ["month"],
    "dayofyear": ["dayofyear"],
}

# Các raw signal external mà ta cho phép dùng để học historical profile.
# Chúng đều là tín hiệu "có thể hợp thức hóa" sang tương lai bằng cách dùng mẫu lịch sử.
SAFE_PROFILE_RAW_COLUMNS = [
    "raw_orders_count",
    "raw_orders_unique_customers",
    "raw_orders_first_time_customers",
    "raw_orders_status_cancelled_count",
    "raw_items_lines_count",
    "raw_items_units_sum",
    "raw_items_discount_sum",
    "raw_items_promo_lines_count",
    "raw_web_sessions_sum",
    "raw_web_unique_visitors_sum",
    "raw_web_page_views_sum",
    "raw_web_bounce_sessions_sum",
    "raw_web_session_duration_total_sum",
    "raw_promo_active_campaigns_count",
    "raw_promo_discount_value_sum",
    "raw_promo_min_order_value_sum",
    "raw_signups_count",
]

CALENDAR_FEATURE_COLUMNS = [
    "time_idx",
    "year",
    "quarter",
    "month",
    "weekofyear",
    "dayofmonth",
    "dayofyear",
    "dow",
    "weekofmonth",
    "days_in_month",
    "month_progress",
    "is_weekend",
    "is_month_start",
    "is_month_end",
    "is_quarter_start",
    "is_quarter_end",
    "is_year_start",
    "is_year_end",
    "is_peak_season",
    "is_valley_season",
    "is_q2",
    "is_q4",
    "month_sin",
    "month_cos",
    "dow_sin",
    "dow_cos",
    "dayofyear_sin",
    "dayofyear_cos",
    "fourier_week_sin_1",
    "fourier_week_cos_1",
    "fourier_week_sin_2",
    "fourier_week_cos_2",
    "fourier_year_sin_1",
    "fourier_year_cos_1",
    "fourier_year_sin_2",
    "fourier_year_cos_2",
]

STATIC_PROFILE_FEATURE_COLUMNS = [
    "Revenue_profile_dow_mean",
    "Revenue_profile_month_mean",
    "Revenue_profile_dayofyear_mean",
    "Revenue_last_train_year_dayofyear",
    "COGS_profile_dow_mean",
    "COGS_profile_month_mean",
    "COGS_profile_dayofyear_mean",
    "COGS_last_train_year_dayofyear",
    "gross_margin_pct_profile_dow_mean",
    "gross_margin_pct_profile_month_mean",
    "gross_margin_pct_profile_dayofyear_mean",
    "gross_margin_pct_last_train_year_dayofyear",
    "raw_orders_count_profile_month_mean",
    "raw_orders_count_profile_dayofyear_mean",
    "raw_orders_unique_customers_profile_dayofyear_mean",
    "raw_orders_first_time_customers_profile_dayofyear_mean",
    "raw_items_lines_count_profile_dayofyear_mean",
    "raw_items_units_sum_profile_dayofyear_mean",
    "raw_items_discount_sum_profile_dayofyear_mean",
    "raw_items_promo_lines_count_profile_dayofyear_mean",
    "raw_web_sessions_sum_profile_month_mean",
    "raw_web_sessions_sum_profile_dayofyear_mean",
    "raw_web_unique_visitors_sum_profile_dayofyear_mean",
    "raw_web_page_views_sum_profile_dayofyear_mean",
    "raw_web_bounce_sessions_sum_profile_dayofyear_mean",
    "raw_web_session_duration_total_sum_profile_dayofyear_mean",
    "raw_promo_active_campaigns_count_profile_dayofyear_mean",
    "raw_promo_discount_value_sum_profile_dayofyear_mean",
    "raw_promo_min_order_value_sum_profile_dayofyear_mean",
    "raw_signups_count_profile_dayofyear_mean",
    "profile_orders_per_visitor_dayofyear",
    "profile_orders_per_visitor_month",
    "profile_pages_per_session_dayofyear",
    "profile_bounce_rate_dayofyear",
    "profile_session_duration_dayofyear",
    "profile_discount_rate_dayofyear",
    "profile_cancel_rate_dayofyear",
    "profile_first_time_customer_share_dayofyear",
    "profile_avg_units_per_order_dayofyear",
    "profile_avg_order_value_dayofyear",
    "profile_avg_cogs_per_order_dayofyear",
    "profile_promo_line_share_dayofyear",
]


def ensure_dir(path: Path) -> None:
    """Tạo thư mục đầu ra nếu chưa tồn tại."""
    path.mkdir(parents=True, exist_ok=True)


def safe_read_csv(path: Path, **kwargs) -> pd.DataFrame:
    """Đọc CSV với `low_memory=False` để tránh suy luận dtype không ổn định."""
    return pd.read_csv(path, low_memory=False, **kwargs)


def slugify(value: object) -> str:
    """Chuẩn hóa tên category / source thành tên cột an toàn."""
    text = re.sub(r"[^0-9a-zA-Z]+", "_", str(value).strip().lower())
    return text.strip("_") or "unknown"


def safe_divide(numerator: pd.Series | np.ndarray, denominator: pd.Series | np.ndarray) -> np.ndarray:
    """Chia an toàn, trả về 0 nếu mẫu số gần 0 để tránh inf / nan."""
    denom = np.asarray(denominator, dtype=float)
    numer = np.asarray(numerator, dtype=float)
    return np.where(np.abs(denom) > 1e-12, numer / denom, 0.0)


def daily_count_crosstab(df: pd.DataFrame, date_col: str, value_col: str, prefix: str) -> pd.DataFrame:
    """
    Đếm số lần xuất hiện theo ngày của một biến phân loại.

    Ví dụ:
    - số đơn theo `order_status`
    - số đơn theo `payment_method`
    - số signup theo `acquisition_channel`
    """
    tab = pd.crosstab(df[date_col], df[value_col], dropna=False)
    tab.columns = [f"{prefix}{slugify(col)}_count" for col in tab.columns]
    return tab


def daily_sum_pivot(
    df: pd.DataFrame,
    date_col: str,
    group_col: str,
    value_col: str,
    prefix: str,
    suffix: str,
) -> pd.DataFrame:
    """
    Pivot theo ngày và cộng dồn một metric số theo từng nhóm.

    Dùng cho các trường hợp như:
    - tổng units theo category mỗi ngày
    - tổng revenue theo category mỗi ngày
    - tổng sessions theo traffic source mỗi ngày
    """
    tab = pd.pivot_table(
        df,
        index=date_col,
        columns=group_col,
        values=value_col,
        aggfunc="sum",
    )
    tab = tab.fillna(0.0)
    tab.columns = [f"{prefix}{slugify(col)}_{suffix}" for col in tab.columns]
    return tab


def add_calendar_features(df: pd.DataFrame, date_col: str = "Date") -> pd.DataFrame:
    """
    Tạo các feature lịch cơ bản và cyclical feature.

    Đây là nhóm feature "biết trước 100%" trong tương lai nên rất an toàn cho forecasting.
    """
    d = df.copy()
    dt = d[date_col]

    # Chỉ số thời gian tuyến tính để model học xu hướng dài hạn.
    d["time_idx"] = (dt - dt.min()).dt.days.astype(int)

    # Các biến lịch rời rạc.
    d["year"] = dt.dt.year
    d["quarter"] = dt.dt.quarter
    d["month"] = dt.dt.month
    d["weekofyear"] = dt.dt.isocalendar().week.astype(int)
    d["dayofmonth"] = dt.dt.day
    d["dayofyear"] = dt.dt.dayofyear
    d["dow"] = dt.dt.dayofweek
    d["weekofmonth"] = ((d["dayofmonth"] - 1) // 7 + 1).astype(int)
    d["days_in_month"] = dt.dt.days_in_month
    d["month_progress"] = safe_divide(d["dayofmonth"], d["days_in_month"])

    # Cờ thời điểm biên để bắt các hiệu ứng cuối tháng / đầu quý / cuối năm.
    d["is_weekend"] = (d["dow"] >= 5).astype(int)
    d["is_month_start"] = dt.dt.is_month_start.astype(int)
    d["is_month_end"] = dt.dt.is_month_end.astype(int)
    d["is_quarter_start"] = dt.dt.is_quarter_start.astype(int)
    d["is_quarter_end"] = dt.dt.is_quarter_end.astype(int)
    d["is_year_start"] = dt.dt.is_year_start.astype(int)
    d["is_year_end"] = dt.dt.is_year_end.astype(int)

    # Cờ mùa vụ rút ra từ EDA trước đó.
    d["is_peak_season"] = d["month"].isin([4, 5, 6]).astype(int)
    d["is_valley_season"] = d["month"].isin([11, 12, 1]).astype(int)
    d["is_q2"] = (d["quarter"] == 2).astype(int)
    d["is_q4"] = (d["quarter"] == 4).astype(int)

    # Biến cyclical giúp model hiểu tính vòng tròn của tháng / thứ / ngày trong năm.
    d["month_sin"] = np.sin(2.0 * np.pi * d["month"] / 12.0)
    d["month_cos"] = np.cos(2.0 * np.pi * d["month"] / 12.0)
    d["dow_sin"] = np.sin(2.0 * np.pi * d["dow"] / 7.0)
    d["dow_cos"] = np.cos(2.0 * np.pi * d["dow"] / 7.0)
    d["dayofyear_sin"] = np.sin(2.0 * np.pi * d["dayofyear"] / 365.25)
    d["dayofyear_cos"] = np.cos(2.0 * np.pi * d["dayofyear"] / 365.25)
    return d


def add_fourier_features(
    df: pd.DataFrame,
    date_col: str = "Date",
    periods: dict[str, float] | None = None,
    order: int = 2,
) -> pd.DataFrame:
    """
    Tạo Fourier terms để mô hình hóa seasonality mượt hơn.

    So với biến lịch rời rạc, Fourier feature giúp model bắt chu kỳ tuần / năm
    theo dạng liên tục và thường hiệu quả với time series.
    """
    d = df.copy()
    if periods is None:
        periods = {"week": 7.0, "year": 365.25}

    t = (d[date_col] - d[date_col].min()).dt.days.astype(float)
    for name, period in periods.items():
        for k in range(1, order + 1):
            angle = 2.0 * np.pi * k * t / period
            d[f"fourier_{name}_sin_{k}"] = np.sin(angle)
            d[f"fourier_{name}_cos_{k}"] = np.cos(angle)
    return d


def add_target_helper_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Tạo các cột hỗ trợ từ target.

    Các cột này chỉ là "biến trung gian" để tạo lag / rolling / profile.
    Chúng sẽ bị loại khỏi bảng cuối cùng vì cùng-ngày sẽ gây leakage.
    """
    d = df.copy()
    d["gross_profit"] = d["Revenue"] - d["COGS"]
    d["gross_margin_pct"] = np.where(
        d["Revenue"].notna() & (np.abs(d["Revenue"]) > 1e-12),
        d["gross_profit"] / d["Revenue"],
        np.nan,
    )
    return d


def add_lag_features(df: pd.DataFrame, columns: list[str], lags: list[int]) -> tuple[pd.DataFrame, list[str]]:
    """Tạo lag feature cho từng cột mục tiêu / helper target."""
    d = df.copy()
    created: list[str] = []
    for col in columns:
        for lag in lags:
            name = f"{col}_lag_{lag}"
            d[name] = d[col].shift(lag)
            created.append(name)
    return d, created


def add_rolling_features(df: pd.DataFrame, columns: list[str], windows: list[int]) -> tuple[pd.DataFrame, list[str]]:
    """
    Tạo rolling statistics trên chuỗi đã shift 1 bước.

    Việc shift trước khi rolling giúp đảm bảo giá trị ngày t không nhìn thấy target của chính ngày t.
    """
    d = df.copy()
    created: list[str] = []
    for col in columns:
        shifted = d[col].shift(1)
        for window in windows:
            roll = shifted.rolling(window=window, min_periods=1)
            for stat_name, stat_series in {
                "mean": roll.mean(),
                "std": roll.std(),
                "min": roll.min(),
                "max": roll.max(),
            }.items():
                name = f"{col}_roll_{stat_name}_{window}"
                d[name] = stat_series
                created.append(name)
    return d, created


def add_ewm_features(df: pd.DataFrame, columns: list[str], spans: list[int]) -> tuple[pd.DataFrame, list[str]]:
    """
    Tạo EWM feature trên chuỗi đã shift 1 bước.

    EWM thường hữu ích để giữ "trí nhớ ngắn hạn" tốt hơn rolling mean đơn thuần.
    """
    d = df.copy()
    created: list[str] = []
    for col in columns:
        shifted = d[col].shift(1)
        for span in spans:
            name = f"{col}_ewm_mean_{span}"
            d[name] = shifted.ewm(span=span, adjust=False).mean()
            created.append(name)
    return d, created


def add_history_profile_features(
    df: pd.DataFrame,
    columns: list[str],
    train_end: pd.Timestamp,
) -> tuple[pd.DataFrame, list[str]]:
    """
    Học historical profile từ giai đoạn train rồi chiếu sang cả train và test.

    Ví dụ:
    - doanh thu trung bình theo thứ trong tuần
    - số đơn trung bình theo ngày trong năm
    - web sessions trung bình theo tháng

    Đây là cách tận dụng external raw signals mà không dùng trực tiếp giá trị
    "cùng ngày tương lai" vốn không thể biết trước.
    """
    d = df.copy()

    # Chỉ dùng lịch sử đến hết train để tránh nhìn vào tương lai.
    hist = d.loc[d["Date"] <= train_end, ["Date", "dow", "month", "dayofyear", *columns]].copy()
    created: list[str] = []

    last_train_year = int(hist["Date"].dt.year.max())
    last_train = hist.loc[hist["Date"].dt.year == last_train_year].copy()

    for col in columns:
        # Profile theo từng khóa lịch.
        for suffix, keys in PROFILE_KEYSETS.items():
            prof = (
                hist.groupby(keys, dropna=False)[col]
                .mean()
                .rename(f"{col}_profile_{suffix}_mean")
                .reset_index()
            )
            d = d.merge(prof, on=keys, how="left")
            created.append(f"{col}_profile_{suffix}_mean")

        # Giữ thêm profile của riêng năm train cuối cùng để bắt "chế độ mới nhất".
        last_year = (
            last_train.groupby("dayofyear", dropna=False)[col]
            .mean()
            .rename(f"{col}_last_train_year_dayofyear")
            .reset_index()
        )
        d = d.merge(last_year, on="dayofyear", how="left")
        created.append(f"{col}_last_train_year_dayofyear")

    return d, created


def add_recursive_interaction_features(df: pd.DataFrame) -> tuple[pd.DataFrame, list[str]]:
    """
    Tạo interaction giữa các lag / target-derived recursive feature.

    Đây là nhóm feature cần inference tuần tự, vì chúng phụ thuộc vào target quá khứ
    hoặc target dự báo ở những bước trước.
    """
    d = df.copy()
    created: list[str] = []

    specs = [
        ("Revenue_lag_1", "Revenue_lag_7", "Revenue_lag_1_to_7_ratio"),
        ("Revenue_lag_7", "Revenue_lag_30", "Revenue_lag_7_to_30_ratio"),
        ("Revenue_lag_1", "Revenue_lag_365", "Revenue_lag_1_to_365_ratio"),
        ("COGS_lag_1", "COGS_lag_7", "COGS_lag_1_to_7_ratio"),
        ("COGS_lag_7", "COGS_lag_30", "COGS_lag_7_to_30_ratio"),
        ("COGS_lag_1", "COGS_lag_365", "COGS_lag_1_to_365_ratio"),
        ("gross_margin_pct_lag_1", "gross_margin_pct_lag_7", "gross_margin_pct_lag_1_minus_7"),
        ("gross_margin_pct_lag_7", "gross_margin_pct_lag_30", "gross_margin_pct_lag_7_minus_30"),
    ]

    for left, right, name in specs:
        if left not in d.columns or right not in d.columns:
            continue

        # Một số feature là tỉ lệ, một số là độ chênh.
        if name.endswith("_minus_7"):
            d[name] = d[left] - d[right]
        else:
            d[name] = safe_divide(d[left], d[right])
        created.append(name)

    return d, created


def add_safe_interaction_features(df: pd.DataFrame) -> tuple[pd.DataFrame, list[str]]:
    """
    Tạo interaction an toàn từ historical profile hoặc inventory state.

    Điểm khác với `add_recursive_interaction_features`:
    - nhóm này không phụ thuộc trực tiếp vào target tương lai chưa biết
    - có thể giữ trong bảng feature cuối cùng như static feature
    """
    d = df.copy()
    created: list[str] = []

    profile_specs = [
        (
            "raw_orders_count_profile_dayofyear_mean",
            "raw_web_unique_visitors_sum_profile_dayofyear_mean",
            "profile_orders_per_visitor_dayofyear",
        ),
        (
            "raw_orders_count_profile_month_mean",
            "raw_web_unique_visitors_sum_profile_month_mean",
            "profile_orders_per_visitor_month",
        ),
        (
            "raw_web_page_views_sum_profile_dayofyear_mean",
            "raw_web_sessions_sum_profile_dayofyear_mean",
            "profile_pages_per_session_dayofyear",
        ),
        (
            "raw_web_bounce_sessions_sum_profile_dayofyear_mean",
            "raw_web_sessions_sum_profile_dayofyear_mean",
            "profile_bounce_rate_dayofyear",
        ),
        (
            "raw_web_session_duration_total_sum_profile_dayofyear_mean",
            "raw_web_sessions_sum_profile_dayofyear_mean",
            "profile_session_duration_dayofyear",
        ),
        (
            "raw_items_discount_sum_profile_dayofyear_mean",
            "Revenue_profile_dayofyear_mean",
            "profile_discount_rate_dayofyear",
        ),
        (
            "raw_orders_status_cancelled_count_profile_dayofyear_mean",
            "raw_orders_count_profile_dayofyear_mean",
            "profile_cancel_rate_dayofyear",
        ),
        (
            "raw_orders_first_time_customers_profile_dayofyear_mean",
            "raw_orders_unique_customers_profile_dayofyear_mean",
            "profile_first_time_customer_share_dayofyear",
        ),
        (
            "raw_items_units_sum_profile_dayofyear_mean",
            "raw_orders_count_profile_dayofyear_mean",
            "profile_avg_units_per_order_dayofyear",
        ),
        (
            "Revenue_profile_dayofyear_mean",
            "raw_orders_count_profile_dayofyear_mean",
            "profile_avg_order_value_dayofyear",
        ),
        (
            "COGS_profile_dayofyear_mean",
            "raw_orders_count_profile_dayofyear_mean",
            "profile_avg_cogs_per_order_dayofyear",
        ),
        (
            "raw_items_promo_lines_count_profile_dayofyear_mean",
            "raw_items_lines_count_profile_dayofyear_mean",
            "profile_promo_line_share_dayofyear",
        ),
    ]

    for left, right, name in profile_specs:
        if left in d.columns and right in d.columns:
            d[name] = safe_divide(d[left], d[right])
            created.append(name)

    # Quy đổi inventory state từ dạng tổng / đếm sang dạng trung bình / tỷ lệ dễ học hơn.
    return d, created


def build_selected_recursive_columns() -> list[str]:
    """Danh sách recursive feature thực sự muốn giữ ở output cuối."""
    selected: list[str] = []

    for col in ["Revenue", "COGS", "gross_margin_pct"]:
        for lag in TARGET_LAGS:
            selected.append(f"{col}_lag_{lag}")

    for col in ["Revenue", "COGS"]:
        for window in TARGET_WINDOWS:
            selected.append(f"{col}_roll_mean_{window}")
            selected.append(f"{col}_roll_std_{window}")

    for window in TARGET_WINDOWS:
        selected.append(f"gross_margin_pct_roll_mean_{window}")

    for col in ["Revenue", "COGS", "gross_margin_pct"]:
        for span in TARGET_EWM_SPANS:
            selected.append(f"{col}_ewm_mean_{span}")

    selected.extend(
        [
            "Revenue_lag_1_to_7_ratio",
            "Revenue_lag_7_to_30_ratio",
            "Revenue_lag_1_to_365_ratio",
            "COGS_lag_1_to_7_ratio",
            "COGS_lag_7_to_30_ratio",
            "COGS_lag_1_to_365_ratio",
            "gross_margin_pct_lag_1_minus_7",
            "gross_margin_pct_lag_7_minus_30",
        ]
    )
    return selected


def select_final_feature_set(
    train_df: pd.DataFrame,
    test_df: pd.DataFrame,
    static_feature_columns: list[str],
    recursive_feature_columns: list[str],
) -> tuple[pd.DataFrame, pd.DataFrame, list[str], list[str]]:
    """
    Chọn bộ feature cuối cùng theo hướng "gọn nhưng mạnh".

    Ý tưởng:
    - giữ toàn bộ calendar / seasonality
    - giữ một nhóm profile feature thật sự có ý nghĩa
    - giữ recursive features cốt lõi cho target
    - bỏ các cột profile quá chi tiết và các cột recursive phụ ít giá trị
    """
    static_keep = [c for c in CALENDAR_FEATURE_COLUMNS + STATIC_PROFILE_FEATURE_COLUMNS if c in static_feature_columns]
    recursive_keep = [c for c in build_selected_recursive_columns() if c in recursive_feature_columns]
    keep_cols = ["Date", "Revenue", "COGS", *static_keep, *recursive_keep]

    train_df = train_df[keep_cols].copy()
    test_df = test_df[keep_cols].copy()
    return train_df, test_df, static_keep, recursive_keep


def build_order_daily_raw_features(
    orders: pd.DataFrame,
    geography: pd.DataFrame,
) -> tuple[pd.DataFrame, list[str]]:
    """
    Tạo bảng raw feature theo ngày từ `orders.csv`.

    Các cột sinh ra là tín hiệu cùng-ngày, ví dụ:
    - số đơn
    - số khách unique
    - mix theo status / payment / source / device / region

    Lưu ý: bảng raw này chỉ dùng để học profile lịch sử, không giữ nguyên ở đầu ra cuối.
    """
    o = orders.copy()
    o["order_date"] = pd.to_datetime(o["order_date"])
    geo = geography[["zip", "region"]].copy()
    o = o.merge(geo, on="zip", how="left")
    o["region"] = o["region"].fillna("unknown")

    # Đánh dấu khách mua lần đầu trong chính ngày đó.
    first_order_dates = o.groupby("customer_id")["order_date"].transform("min")
    o["is_first_order"] = (o["order_date"] == first_order_dates).astype(int)

    daily = o.groupby("order_date").agg(
        raw_orders_count=("order_id", "count"),
        raw_orders_unique_customers=("customer_id", "nunique"),
        raw_orders_first_time_customers=("is_first_order", "sum"),
    )

    parts = [
        daily_count_crosstab(o, "order_date", "order_status", "raw_orders_status_"),
        daily_count_crosstab(o, "order_date", "payment_method", "raw_orders_payment_"),
        daily_count_crosstab(o, "order_date", "order_source", "raw_orders_source_"),
        daily_count_crosstab(o, "order_date", "device_type", "raw_orders_device_"),
        daily_count_crosstab(o, "order_date", "region", "raw_orders_region_"),
    ]

    out = daily.join(parts, how="left").reset_index().rename(columns={"order_date": "Date"})
    value_cols = [c for c in out.columns if c != "Date"]
    return out, value_cols


def build_order_item_daily_raw_features(
    order_items: pd.DataFrame,
    orders: pd.DataFrame,
    products: pd.DataFrame,
) -> tuple[pd.DataFrame, list[str]]:
    """
    Tạo raw feature theo ngày từ `order_items.csv` + `orders.csv` + `products.csv`.

    Nhóm này chứa nhiều tín hiệu mạnh:
    - quantity
    - revenue proxy
    - discount
    - cogs
    - category / segment mix
    """
    oi = order_items.copy()
    o = orders[["order_id", "order_date"]].copy()
    p = products[["product_id", "category", "segment", "price", "cogs"]].copy()

    o["order_date"] = pd.to_datetime(o["order_date"])
    oi = oi.merge(o, on="order_id", how="left")
    oi = oi.merge(p, on="product_id", how="left")

    # `item_target_revenue` dùng đúng định nghĩa Revenue mục tiêu trong `sales.csv`.
    oi["item_target_revenue"] = oi["quantity"] * oi["unit_price"]

    # `item_payment_proxy` gần với số tiền khách trả thực tế sau discount.
    oi["item_payment_proxy"] = oi["item_target_revenue"] - oi["discount_amount"]

    # Giá trị theo catalog và COGS để suy ra discount / margin.
    oi["item_catalog_value"] = oi["quantity"] * oi["price"]
    oi["item_cogs_proxy"] = oi["quantity"] * oi["cogs"]
    oi["item_margin_proxy"] = oi["item_target_revenue"] - oi["item_cogs_proxy"]
    oi["item_has_promo"] = oi["promo_id"].notna().astype(int)
    oi["item_has_second_promo"] = oi["promo_id_2"].notna().astype(int)

    daily = oi.groupby("order_date").agg(
        raw_items_lines_count=("product_id", "count"),
        raw_items_units_sum=("quantity", "sum"),
        raw_items_distinct_products_count=("product_id", "nunique"),
        raw_items_target_revenue_sum=("item_target_revenue", "sum"),
        raw_items_payment_proxy_sum=("item_payment_proxy", "sum"),
        raw_items_discount_sum=("discount_amount", "sum"),
        raw_items_catalog_value_sum=("item_catalog_value", "sum"),
        raw_items_cogs_sum=("item_cogs_proxy", "sum"),
        raw_items_margin_sum=("item_margin_proxy", "sum"),
        raw_items_promo_lines_count=("item_has_promo", "sum"),
        raw_items_stack_promo_lines_count=("item_has_second_promo", "sum"),
    )

    parts = [
        daily_sum_pivot(oi, "order_date", "category", "quantity", "raw_items_category_", "units_sum"),
        daily_sum_pivot(oi, "order_date", "category", "item_target_revenue", "raw_items_category_", "revenue_sum"),
        daily_sum_pivot(oi, "order_date", "segment", "quantity", "raw_items_segment_", "units_sum"),
    ]

    out = daily.join(parts, how="left").reset_index().rename(columns={"order_date": "Date"})
    value_cols = [c for c in out.columns if c != "Date"]
    return out, value_cols


def build_payment_daily_raw_features(
    payments: pd.DataFrame,
    orders: pd.DataFrame,
) -> tuple[pd.DataFrame, list[str]]:
    """Tạo raw feature theo ngày từ `payments.csv`."""
    p = payments.copy()
    o = orders[["order_id", "order_date"]].copy()
    o["order_date"] = pd.to_datetime(o["order_date"])
    p = p.merge(o, on="order_id", how="left")
    p["installment_gt1"] = (p["installments"] > 1).astype(int)

    daily = p.groupby("order_date").agg(
        raw_payments_count=("order_id", "count"),
        raw_payments_value_sum=("payment_value", "sum"),
        raw_payments_installments_sum=("installments", "sum"),
        raw_payments_installment_gt1_count=("installment_gt1", "sum"),
    )

    out = daily.reset_index().rename(columns={"order_date": "Date"})
    value_cols = [c for c in out.columns if c != "Date"]
    return out, value_cols


def build_shipment_daily_raw_features(shipments: pd.DataFrame) -> tuple[pd.DataFrame, list[str]]:
    """
    Tạo raw feature theo ngày ship từ `shipments.csv`.

    Đây là tín hiệu hậu nghiệm nếu xét đúng ngày tương lai, nên cũng chỉ dùng để học profile.
    """
    s = shipments.copy()
    s["ship_date"] = pd.to_datetime(s["ship_date"])
    s["delivery_date"] = pd.to_datetime(s["delivery_date"])
    s["delivery_days"] = (s["delivery_date"] - s["ship_date"]).dt.days
    s["is_fast_delivery"] = (s["delivery_days"] <= 2).astype(int)
    s["is_slow_delivery"] = (s["delivery_days"] >= 6).astype(int)

    daily = s.groupby("ship_date").agg(
        raw_shipments_count=("order_id", "count"),
        raw_shipments_fee_sum=("shipping_fee", "sum"),
        raw_shipments_delivery_days_sum=("delivery_days", "sum"),
        raw_shipments_fast_count=("is_fast_delivery", "sum"),
        raw_shipments_slow_count=("is_slow_delivery", "sum"),
    )

    out = daily.reset_index().rename(columns={"ship_date": "Date"})
    value_cols = [c for c in out.columns if c != "Date"]
    return out, value_cols


def build_return_daily_raw_features(returns: pd.DataFrame) -> tuple[pd.DataFrame, list[str]]:
    """Tạo raw feature theo ngày từ `returns.csv`, gồm số lượng và lý do trả hàng."""
    r = returns.copy()
    r["return_date"] = pd.to_datetime(r["return_date"])

    daily = r.groupby("return_date").agg(
        raw_returns_count=("return_id", "count"),
        raw_returns_qty_sum=("return_quantity", "sum"),
        raw_returns_refund_sum=("refund_amount", "sum"),
    )
    reason_mix = daily_count_crosstab(r, "return_date", "return_reason", "raw_returns_reason_")

    out = daily.join(reason_mix, how="left").reset_index().rename(columns={"return_date": "Date"})
    value_cols = [c for c in out.columns if c != "Date"]
    return out, value_cols


def build_review_daily_raw_features(reviews: pd.DataFrame) -> tuple[pd.DataFrame, list[str]]:
    """Tạo raw feature theo ngày từ `reviews.csv`, gồm số review và phân bố rating."""
    rv = reviews.copy()
    rv["review_date"] = pd.to_datetime(rv["review_date"])
    rv["is_4plus"] = (rv["rating"] >= 4).astype(int)
    rv["is_2minus"] = (rv["rating"] <= 2).astype(int)

    daily = rv.groupby("review_date").agg(
        raw_reviews_count=("review_id", "count"),
        raw_reviews_rating_sum=("rating", "sum"),
        raw_reviews_4plus_count=("is_4plus", "sum"),
        raw_reviews_2minus_count=("is_2minus", "sum"),
    )

    out = daily.reset_index().rename(columns={"review_date": "Date"})
    value_cols = [c for c in out.columns if c != "Date"]
    return out, value_cols


def build_web_daily_raw_features(web: pd.DataFrame) -> tuple[pd.DataFrame, list[str]]:
    """
    Tạo raw feature theo ngày từ `web_traffic.csv`.

    Ngoài các cột gốc, hàm này còn tạo:
    - bounce sessions = sessions * bounce_rate
    - session_duration_total = sessions * avg_session_duration_sec

    Hai cột này giúp tạo tỷ lệ / trung bình có trọng số sau này.
    """
    w = web.copy()
    w["date"] = pd.to_datetime(w["date"])
    w["bounce_sessions"] = w["sessions"] * w["bounce_rate"]
    w["session_duration_total"] = w["sessions"] * w["avg_session_duration_sec"]

    daily = w.groupby("date").agg(
        raw_web_sessions_sum=("sessions", "sum"),
        raw_web_unique_visitors_sum=("unique_visitors", "sum"),
        raw_web_page_views_sum=("page_views", "sum"),
        raw_web_bounce_sessions_sum=("bounce_sessions", "sum"),
        raw_web_session_duration_total_sum=("session_duration_total", "sum"),
    )
    source_mix = daily_sum_pivot(w, "date", "traffic_source", "sessions", "raw_web_source_", "sessions_sum")

    out = daily.join(source_mix, how="left").reset_index().rename(columns={"date": "Date"})
    value_cols = [c for c in out.columns if c != "Date"]
    return out, value_cols


def build_promo_daily_raw_features(promotions: pd.DataFrame) -> tuple[pd.DataFrame, list[str]]:
    """
    Bung lịch active của promotion thành bảng theo ngày.

    Kết quả là số campaign đang chạy, loại promo, mức discount, min order value...
    """
    p = promotions.copy()
    p["start_date"] = pd.to_datetime(p["start_date"])
    p["end_date"] = pd.to_datetime(p["end_date"])

    rows: list[dict[str, float | pd.Timestamp]] = []
    for _, row in p.iterrows():
        active_dates = pd.date_range(row["start_date"], row["end_date"], freq="D")
        if len(active_dates) == 0:
            continue

        promo_type = slugify(row.get("promo_type", "unknown"))
        for dt in active_dates:
            rows.append(
                {
                    "Date": dt,
                    "raw_promo_active_campaigns_count": 1,
                    "raw_promo_percentage_count": int(promo_type == "percentage"),
                    "raw_promo_fixed_count": int(promo_type == "fixed"),
                    "raw_promo_stackable_count": int(row.get("stackable_flag", 0) == 1),
                    "raw_promo_discount_value_sum": float(row.get("discount_value", 0) or 0.0),
                    "raw_promo_min_order_value_sum": float(row.get("min_order_value", 0) or 0.0),
                    "raw_promo_global_count": int(pd.isna(row.get("applicable_category"))),
                    "raw_promo_category_specific_count": int(pd.notna(row.get("applicable_category"))),
                }
            )

    if not rows:
        return pd.DataFrame({"Date": []}), []

    daily = pd.DataFrame(rows).groupby("Date", as_index=False).sum()
    value_cols = [c for c in daily.columns if c != "Date"]
    return daily, value_cols


def build_customer_signup_daily_raw_features(customers: pd.DataFrame) -> tuple[pd.DataFrame, list[str]]:
    """Tạo raw feature signup theo ngày từ `customers.csv`."""
    c = customers.copy()
    c["signup_date"] = pd.to_datetime(c["signup_date"])
    c["acquisition_channel"] = c["acquisition_channel"].fillna("unknown")

    daily = c.groupby("signup_date").agg(
        raw_signups_count=("customer_id", "count"),
    )
    acq_mix = daily_count_crosstab(c, "signup_date", "acquisition_channel", "raw_signups_channel_")

    out = daily.join(acq_mix, how="left").reset_index().rename(columns={"signup_date": "Date"})
    value_cols = [c for c in out.columns if c != "Date"]
    return out, value_cols


def build_inventory_daily_state_features(
    inventory: pd.DataFrame,
    full_dates: pd.DataFrame,
) -> tuple[pd.DataFrame, list[str]]:
    """
    Chuyển snapshot tồn kho cuối tháng thành "state feature" theo ngày.

    Logic:
    - aggregate inventory theo ngày snapshot
    - merge vào full calendar
    - forward fill để mỗi ngày dùng trạng thái inventory gần nhất đã biết

    Đây là biến dạng "last known state", hợp lý hơn việc gán 0 cho tương lai.
    """
    inv = inventory.copy()
    inv["snapshot_date"] = pd.to_datetime(inv["snapshot_date"])

    monthly = inv.groupby("snapshot_date").agg(
        inv_stock_on_hand_sum=("stock_on_hand", "sum"),
        inv_units_received_sum=("units_received", "sum"),
        inv_units_sold_sum=("units_sold", "sum"),
        inv_stockout_days_sum=("stockout_days", "sum"),
        inv_days_of_supply_sum=("days_of_supply", "sum"),
        inv_fill_rate_sum=("fill_rate", "sum"),
        inv_stockout_flag_sum=("stockout_flag", "sum"),
        inv_overstock_flag_sum=("overstock_flag", "sum"),
        inv_reorder_flag_sum=("reorder_flag", "sum"),
        inv_sell_through_rate_sum=("sell_through_rate", "sum"),
        inv_rows_count=("product_id", "count"),
    )

    state = full_dates.copy()
    state = state.merge(
        monthly.reset_index().rename(columns={"snapshot_date": "Date"}),
        on="Date",
        how="left",
    ).sort_values("Date")

    value_cols = [c for c in state.columns if c != "Date"]
    state[value_cols] = state[value_cols].ffill()
    state = state.rename(columns={c: f"state_{c}" for c in value_cols})
    state_cols = [c for c in state.columns if c != "Date"]
    return state, state_cols


def build_base_frame(sales: pd.DataFrame, sample_submission: pd.DataFrame) -> tuple[pd.DataFrame, pd.Timestamp]:
    """
    Tạo calendar đầy đủ từ ngày train đầu tiên đến ngày test cuối cùng.

    Bảng này là trục xương sống của toàn bộ pipeline.
    """
    s = sales.copy()
    sample = sample_submission.copy()
    s["Date"] = pd.to_datetime(s["Date"])
    sample["Date"] = pd.to_datetime(sample["Date"])

    train_start = s["Date"].min()
    test_end = sample["Date"].max()
    full_dates = pd.DataFrame({"Date": pd.date_range(train_start, test_end, freq="D")})
    full = full_dates.merge(s[["Date", "Revenue", "COGS"]], on="Date", how="left")
    return full, pd.Timestamp(s["Date"].max())


def merge_raw_feature_frames(
    base_dates: pd.DataFrame,
    frames: list[tuple[pd.DataFrame, list[str]]],
) -> tuple[pd.DataFrame, list[str]]:
    """
    Merge nhiều bảng raw feature theo ngày thành một bảng chung.

    Mặc định raw feature sẽ được fill 0 vì đây là các đại lượng đếm / tổng trong ngày.
    """
    merged = base_dates.copy()
    raw_cols: list[str] = []

    for frame, value_cols in frames:
        if frame.empty:
            continue
        merged = merged.merge(frame, on="Date", how="left")
        raw_cols.extend(value_cols)

    raw_cols = sorted(set(c for c in raw_cols if c in merged.columns))
    if raw_cols:
        merged[raw_cols] = merged[raw_cols].fillna(0.0)
    return merged, raw_cols


def finalize_features(
    full: pd.DataFrame,
    train_end: pd.Timestamp,
    sample_submission: pd.DataFrame,
    recursive_feature_columns: list[str],
    dropped_same_day_columns: list[str],
) -> tuple[pd.DataFrame, pd.DataFrame, list[str], list[str]]:
    """
    Tạo bảng train / test cuối cùng dùng cho modeling.

    Các bước chính:
    1. Bỏ các cột same-day raw để tránh train-test mismatch.
    2. Tách train / test theo danh sách ngày trong sample submission.
    3. Chia feature thành:
       - static: có thể dùng trực tiếp
       - recursive: cần cập nhật tuần tự khi suy luận
    4. Chỉ fill median cho static feature.

    Chủ đích:
    - Không "làm giả" recursive feature ở test bằng median.
    - Để downstream inference code tự cập nhật lag / rolling / ewm theo dự báo.
    """
    d = full.copy()
    drop_cols = [c for c in dropped_same_day_columns if c in d.columns]
    if drop_cols:
        d = d.drop(columns=drop_cols)

    d = d.sort_values("Date").reset_index(drop=True)
    sample_dates = pd.to_datetime(sample_submission["Date"])
    is_test = d["Date"].isin(sample_dates)

    train_df = d.loc[~is_test].copy()
    train_df = train_df[train_df["Date"] <= train_end].copy()
    test_df = d.loc[is_test].copy().sort_values("Date")

    feature_cols = [c for c in train_df.columns if c not in ["Date", "Revenue", "COGS"]]
    recursive_set = {c for c in recursive_feature_columns if c in feature_cols}
    static_cols = sorted([c for c in feature_cols if c not in recursive_set])
    recursive_cols = sorted(recursive_set)

    # Chỉ static feature mới được fill median ở đây.
    for col in static_cols:
        if not pd.api.types.is_numeric_dtype(train_df[col]):
            continue
        med = train_df[col].median()
        if pd.isna(med):
            med = 0.0
        train_df[col] = train_df[col].fillna(med)
        test_df[col] = test_df[col].fillna(med)

    train_df = train_df.dropna(subset=["Revenue", "COGS"]).reset_index(drop=True)
    test_df = test_df.reset_index(drop=True)
    test_df["Revenue"] = np.nan
    test_df["COGS"] = np.nan

    ordered_test = pd.DataFrame({"Date": pd.to_datetime(sample_submission["Date"])})
    test_df = ordered_test.merge(test_df, on="Date", how="left")
    return train_df, test_df, static_cols, recursive_cols


def save_outputs(
    train_df: pd.DataFrame,
    test_df: pd.DataFrame,
    out_dir: Path,
    metadata: dict,
) -> None:
    """Lưu train/test feature table và metadata mô tả pipeline."""
    ensure_dir(out_dir)
    train_path = out_dir / "train_features.csv"
    test_path = out_dir / "test_features.csv"
    meta_path = out_dir / "feature_pipeline_metadata.json"

    train_df.to_csv(train_path, index=False)
    test_df.to_csv(test_path, index=False)

    with open(meta_path, "w", encoding="utf-8") as f:
        json.dump(metadata, f, indent=2, ensure_ascii=False)

    print(f"Saved: {train_path}")
    print(f"Saved: {test_path}")
    print(f"Saved: {meta_path}")


def build_feature_pipeline(data_dir: Path, out_dir: Path) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Hàm chính điều phối toàn bộ pipeline.

    Luồng xử lý:
    1. Đọc dữ liệu.
    2. Tạo khung ngày đầy đủ.
    3. Tạo helper target, feature lịch, Fourier.
    4. Tạo raw feature từ nhiều bảng dữ liệu.
    5. Biến raw feature thành profile lịch sử an toàn.
    6. Tạo recursive feature từ target lịch sử.
    7. Tách train / test và lưu output.
    """
    sales = safe_read_csv(data_dir / "sales.csv", parse_dates=["Date"])
    sample_submission = safe_read_csv(data_dir / "sample_submission.csv", parse_dates=["Date"])

    orders = safe_read_csv(data_dir / "orders.csv", parse_dates=["order_date"])
    order_items = safe_read_csv(data_dir / "order_items.csv")
    products = safe_read_csv(data_dir / "products.csv")
    web = safe_read_csv(data_dir / "web_traffic.csv", parse_dates=["date"])
    promotions = safe_read_csv(data_dir / "promotions.csv", parse_dates=["start_date", "end_date"])
    customers = safe_read_csv(data_dir / "customers.csv", parse_dates=["signup_date"])
    geography = safe_read_csv(data_dir / "geography.csv")

    full, train_end = build_base_frame(sales, sample_submission)
    full = add_target_helper_features(full)
    full = add_calendar_features(full, date_col="Date")
    full = add_fourier_features(full, date_col="Date", periods={"week": 7.0, "year": 365.25}, order=2)

    # Tạo raw feature cùng-ngày từ từng bảng nguồn.
    raw_frames = [
        build_order_daily_raw_features(orders, geography),
        build_order_item_daily_raw_features(order_items, orders, products),
        build_web_daily_raw_features(web),
        build_promo_daily_raw_features(promotions),
        build_customer_signup_daily_raw_features(customers),
    ]

    # Merge raw feature vào full frame để tiếp tục học profile lịch sử.
    raw_daily, raw_feature_columns = merge_raw_feature_frames(full[["Date"]], raw_frames)
    full = full.merge(raw_daily, on="Date", how="left")

    # Inventory được xử lý riêng dưới dạng "last known state".

    # Học profile lịch sử trên toàn bộ target helper + external raw signals.
    safe_profile_raw_columns = [c for c in SAFE_PROFILE_RAW_COLUMNS if c in raw_feature_columns]
    profile_columns = TARGET_BASE_COLUMNS + safe_profile_raw_columns
    full, profile_feature_columns = add_history_profile_features(full, columns=profile_columns, train_end=train_end)

    # Tạo feature autoregressive từ target lịch sử.
    full, lag_cols = add_lag_features(full, columns=TARGET_BASE_COLUMNS, lags=TARGET_LAGS)
    full, rolling_cols = add_rolling_features(full, columns=TARGET_BASE_COLUMNS, windows=TARGET_WINDOWS)
    full, ewm_cols = add_ewm_features(full, columns=TARGET_BASE_COLUMNS, spans=TARGET_EWM_SPANS)
    full, recursive_interaction_cols = add_recursive_interaction_features(full)

    # Tạo interaction an toàn từ profile / state.
    full, safe_interaction_cols = add_safe_interaction_features(full)

    # Recursive feature là những cột cần được cập nhật theo bước khi infer.
    recursive_feature_columns = lag_cols + rolling_cols + ewm_cols + recursive_interaction_cols

    # Các cột raw cùng-ngày và helper target cùng-ngày không được giữ lại ở output cuối.
    dropped_same_day_columns = ["gross_profit", "gross_margin_pct"] + raw_feature_columns

    train_df, test_df, static_feature_columns, recursive_feature_columns = finalize_features(
        full,
        train_end=train_end,
        sample_submission=sample_submission,
        recursive_feature_columns=recursive_feature_columns,
        dropped_same_day_columns=dropped_same_day_columns,
    )

    train_df, test_df, static_feature_columns, recursive_feature_columns = select_final_feature_set(
        train_df,
        test_df,
        static_feature_columns,
        recursive_feature_columns,
    )

    feature_columns = [c for c in train_df.columns if c not in ["Date", "Revenue", "COGS"]]
    selected_profile_columns = [c for c in static_feature_columns if c not in CALENDAR_FEATURE_COLUMNS]
    metadata = {
        "train_rows": int(len(train_df)),
        "test_rows": int(len(test_df)),
        "n_columns_train": int(train_df.shape[1]),
        "n_columns_test": int(test_df.shape[1]),
        "n_feature_columns": int(len(feature_columns)),
        "train_start": str(train_df["Date"].min().date()),
        "train_end": str(train_df["Date"].max().date()),
        "test_start": str(test_df["Date"].min().date()),
        "test_end": str(test_df["Date"].max().date()),
        "feature_columns": feature_columns,
        "static_feature_columns": static_feature_columns,
        "recursive_feature_columns": recursive_feature_columns,
        "profile_feature_columns_generated": profile_feature_columns,
        "profile_feature_columns_selected": selected_profile_columns,
        "dropped_same_day_columns": dropped_same_day_columns,
        "train_recursive_missing_total": int(train_df[recursive_feature_columns].isna().sum().sum()) if recursive_feature_columns else 0,
        "test_recursive_missing_total": int(test_df[recursive_feature_columns].isna().sum().sum()) if recursive_feature_columns else 0,
        "notes": [
            "The final feature set is intentionally conservative: it keeps calendar features, selected historical profiles, and core recursive target features only.",
            "Same-day raw business signals from orders, items, promotions, web traffic, and signups are used only to learn historical profiles, then removed from the final model-ready table.",
            "Payments, shipments, returns, reviews, and inventory are excluded from the final feature set to reduce train-test mismatch and avoid weakly justified future assumptions.",
            "Recursive target features (lag / rolling / ewm) remain in the output. For inference, downstream model code should update them step-by-step using predictions rather than median-imputing them.",
        ],
    }

    save_outputs(train_df, test_df, out_dir, metadata)
    return train_df, test_df


def parse_args() -> argparse.Namespace:
    """Khai báo tham số CLI cho script."""
    parser = argparse.ArgumentParser(description="Build forecasting-ready daily features for Datathon 2026")
    parser.add_argument(
        "--data_dir",
        type=str,
        default="/kaggle/input/competitions/datathon-2026-round-1",
        help="Folder containing CSV files",
    )
    parser.add_argument(
        "--out_dir",
        type=str,
        default="/kaggle/working/",
        help="Folder to save feature tables",
    )
    args, _ = parser.parse_known_args()

    import os
    if os.path.exists("/kaggle/input"):
        args.data_dir = "/kaggle/input/competitions/datathon-2026-round-1"
        args.out_dir = "/kaggle/working/"

    return args


if __name__ == "__main__":
    args = parse_args()
    build_feature_pipeline(Path(args.data_dir), Path(args.out_dir))


Saved: /kaggle/working/train_features.csv
Saved: /kaggle/working/test_features.csv
Saved: /kaggle/working/feature_pipeline_metadata.json
